# Fase 4: profesor propio en GPU (LetraDeMédico)

Genera respuestas de referencia para el dataset SFT con **Qwen2.5-14B-Instruct-AWQ** (licencia Apache 2.0) corriendo en la GPU de Colab, sin cuotas de API. Usa el mismo prompt y las mismas verificaciones que los profesores de Groq, y recorre los casos **desde el final** (Groq va desde el principio).

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*.

Las respuestas se guardan en **Google Drive** (`MyDrive/letrademedico/respuestas`) después de cada lote. Si Colab se desconecta, vuelve a correr las celdas 1 a 3 y luego la del profesor: continúa donde quedó.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/letrademedico'
# Cuántos casos nuevos hacer: con ~210 ejemplos ya listos, 400 más alcanzan para llegar a ~500 limpios
LIMITE = 400
!mkdir -p {DRIVE}/respuestas

In [ ]:
%cd /content
!test -d Proyecto_LaLetraDeMedico || git clone https://github.com/Valero25/Proyecto_LaLetraDeMedico.git
%cd /content/Proyecto_LaLetraDeMedico
!git pull -q && git log --oneline -1
!wc -l data/generador/casos.jsonl data/generador/respuestas/*.jsonl

## Opción A: vLLM (rápida)

vLLM procesa 64 casos a la vez. Se instala una versión compatible con la T4 (tarda unos minutos). Si esta celda falla (por ejemplo, por incompatibilidad con la GPU o falta de memoria), usa la opción B.

In [ ]:
!pip install -q vllm==0.8.5
!python training/generador/profesor_local.py --motor vllm --limite {LIMITE} --salida {DRIVE}/respuestas

## Opción B: transformers (más lenta, siempre funciona)

Usa **Qwen2.5-7B-Instruct** cuantizado a 4 bits, en lotes de 8. Es más lento que vLLM, pero no depende de su compatibilidad con la GPU. Salta los casos que la opción A ya haya respondido.

In [ ]:
!pip install -q -U bitsandbytes
!python training/generador/profesor_local.py --motor transformers --modelo Qwen/Qwen2.5-7B-Instruct --limite {LIMITE} --salida {DRIVE}/respuestas

## Descargar las respuestas

Cópialas a `data/generador/respuestas/` en el proyecto local (o avísale a Claude dónde quedaron).

In [ ]:
!ls -lh {DRIVE}/respuestas
from google.colab import files
import glob
for archivo in glob.glob(f'{DRIVE}/respuestas/*.jsonl'):
    files.download(archivo)